# Aula 1 — Otimização e Avaliação de Modelos
## Seleção de Features (Feature Selection)

Este notebook reconstrói, em ordem lógica e executável célula a célula, o conteúdo dos
slides 28 a 60 da Aula 1. Cada seção traz uma explicação curta do conceito seguida do
código correspondente, para você rodar e observar o resultado antes de avançar.

**Dataset usado em toda a aula:** `load_breast_cancer` (sklearn) — classificação binária
(tumor maligno/benigno) com 30 features numéricas contínuas. É um bom dataset didático
porque tem features claramente redundantes (ex: `mean radius`, `worst radius`, `radius error`
medem coisas parecidas), o que torna a seleção de features visível na prática.

### Roteiro da aula

1. **Métodos Filter** — avaliam cada feature isoladamente com estatística pura (rápidos, não usam modelo).
2. **Pipeline, boas práticas e comparação de filtros** — como aplicar filtros corretamente sem vazar dados.
3. **Métodos Wrapper** — usam um modelo real, retreinado repetidamente (RFE, RFECV).
4. **Métodos Embedded** — a seleção nasce do próprio treinamento (L1/Lasso, árvores, permutation importance).
5. **Comparação final** — acurácia vs. tempo entre as três famílias.

> **Nota sobre os arquivos originais:** os scripts `slideXX_*.py` desta pasta foram escritos
> como trechos incrementais de slide (cada um assume variáveis já criadas pelo anterior) e
> não rodam sozinhos com `python slideXX.py`. Este notebook resolve isso: cada célula já
> recebe o que precisa da célula anterior, na ordem certa.

## 0. Setup do ambiente

Carregamos o dataset, fazemos o split treino/teste e padronizamos as features com
`StandardScaler`. **Tudo o que vier depois reaproveita `X_train`, `X_test`, `y_train`,
`y_test`, `X_train_sc` e `X_test_sc` definidos aqui** — é o que permite rodar o notebook
célula por célula sem `NameError`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import (
    SelectKBest, f_classif, chi2, mutual_info_classif,
    VarianceThreshold, RFE, RFECV, SelectFromModel
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Shape completo: {X.shape}')
print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

---
# Bloco 1 — Métodos Filter: fundamentos estatísticos

Métodos **filter** avaliam cada feature isoladamente (ou em relação ao alvo `y`) usando uma
métrica estatística, **sem treinar nenhum modelo de ML**. São o passo mais barato e rápido
de seleção de features, mas ignoram interações entre variáveis.

## 1.1 Correlação de Pearson (slide 28)

Mede a correlação **linear** entre cada feature e o alvo. Quanto maior `|r|`, mais a feature
varia linearmente junto com a classe. É o método mais simples e intuitivo — mas só enxerga
relações lineares.

In [ ]:
corr_pearson = X.corrwith(y).abs().sort_values(ascending=False)
print(corr_pearson.head(10))

## 1.2 Correlação de Spearman (slide 29)

Usa o **ranking** dos valores em vez do valor bruto, o que captura relações **monotônicas
não-lineares** (não só lineares) e é mais robusto a outliers. Aqui filtramos apenas as
features com correlação forte (`|corr| > 0.5`).

In [ ]:
from scipy.stats import spearmanr

print('Features com correlação de Spearman forte (|r| > 0.5):\n')
for col in X.columns:
    corr, pval = spearmanr(X[col], y)
    if abs(corr) > 0.5:
        print(f'{col}: r={corr:.3f}, p={pval:.2e}')

## 1.3 Teste Qui-Quadrado — Chi² (slide 30)

Mede a dependência estatística entre variáveis e a classe. **Atenção:** o `chi2` do sklearn
exige valores não-negativos — por isso usamos `MinMaxScaler` (que leva tudo para `[0, 1]`)
em vez do `StandardScaler` (que gera valores negativos).

In [ ]:
mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train)

sel_chi2 = SelectKBest(chi2, k=10)
sel_chi2.fit_transform(X_train_mm, y_train)

selected_idx = sel_chi2.get_support(indices=True)
print('Features selecionadas (Chi²):', X.columns[selected_idx].tolist())

## 1.4 Mutual Information (slide 31)

Mede a redução de incerteza sobre `y` ao conhecer uma feature — captura dependências
**não-lineares e não-monotônicas**, sendo mais geral que Pearson e Spearman (baseado em
teoria da informação).

In [ ]:
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print(mi_series.head(10))

## 1.5 Variance Threshold (slide 32)

Diferente dos métodos anteriores, **não olha para o alvo `y`** — é não-supervisionado.
Remove features com variância muito baixa (quase constantes), que carregam pouca
informação discriminativa. Costuma ser o primeiro filtro de um pipeline.

In [ ]:
var_selector = VarianceThreshold(threshold=0.1)
X_high_var = var_selector.fit_transform(X_train_sc)

removed = X.columns[~var_selector.get_support()]
print(f'Features originais: {X_train_sc.shape[1]}')
print(f'Features após filtro: {X_high_var.shape[1]}')
print(f'Features removidas: {removed.tolist()}')

## 1.6 SelectKBest — consolidando os métodos (slide 33)

`SelectKBest` é a interface genérica do sklearn: aceita qualquer função de score
(`f_classif`/ANOVA, `chi2`, `mutual_info_classif`, ...) e sempre seleciona as `k` melhores
features segundo o critério escolhido. A escolha do teste depende do tipo de dado:

- **ANOVA F-test** (`f_classif`): features contínuas → alvo categórico.
- **Chi²**: features categóricas/não-negativas.
- **Mutual Information**: relações gerais, sem suposição de linearidade.

In [ ]:
sel_anova = SelectKBest(f_classif, k=10)
X_anova = sel_anova.fit_transform(X_train_sc, y_train)

top10_anova = pd.Series(sel_anova.scores_, index=X.columns).sort_values(ascending=False).head(10)
print('Top 10 features (ANOVA):', top10_anova.index.tolist())

---
# Bloco 2 — Pipeline completo, boas práticas e comparação de filtros

Agora saímos da teoria isolada para um fluxo real de treino/teste: como aplicar filtros
sem vazar dados, visualizar os resultados, comparar métodos e combiná-los.

## 2.1 Exemplo completo: pipeline filter (slide 35)

Template que será reaproveitado no restante da aula: split → `StandardScaler` (fit só no
treino) → `SelectKBest(f_classif, k=10)` → `LogisticRegression` → avaliação no teste.

In [ ]:
sel = SelectKBest(f_classif, k=10)
X_tr_sel = sel.fit_transform(X_train_sc, y_train)
X_te_sel = sel.transform(X_test_sc)

clf = LogisticRegression(max_iter=10000)
clf.fit(X_tr_sel, y_train)
acc_filter_pipeline = accuracy_score(y_test, clf.predict(X_te_sel))
print(f'Acurácia: {acc_filter_pipeline:.4f}')

## 2.2 Visualizando os scores (slide 36)

Gráfico de barras horizontais com os 15 maiores F-scores do ANOVA (`sel.scores_`) —
torna tangível quais features dominam o critério estatístico escolhido.

In [ ]:
scores = sel.scores_
idx_sorted = np.argsort(scores)[-15:]

plt.figure(figsize=(8, 6))
plt.barh(range(len(idx_sorted)), scores[idx_sorted])
plt.yticks(range(len(idx_sorted)), X.columns[idx_sorted])
plt.xlabel('F-score (ANOVA)')
plt.title('Top 15 features por F-score')
plt.tight_layout()
plt.show()

## 2.3 Cuidado: Data Leakage (slide 37)

Regra de ouro: `fit_transform` **apenas no treino**, `transform` (sem fit) no teste. Se você
ajustar o seletor de features (ou o scaler) em treino+teste juntos, informação do teste
"vaza" para a etapa de seleção, e a métrica final fica artificialmente otimista.

Abaixo comparamos as duas abordagens lado a lado. Neste dataset a diferença numérica
pode ser pequena (é um dataset limpo e bem separável), mas o mecanismo do vazamento é
real e cresce com datasets menores/mais ruidosos — o problema é metodológico, não apenas
de acurácia.

In [ ]:
# CORRETO: fit apenas no treino
sel_ok = SelectKBest(f_classif, k=10)
Xtr_ok = sel_ok.fit_transform(X_train_sc, y_train)
Xte_ok = sel_ok.transform(X_test_sc)
clf_ok = LogisticRegression(max_iter=10000).fit(Xtr_ok, y_train)
acc_ok = accuracy_score(y_test, clf_ok.predict(Xte_ok))

# ERRADO: fit em todos os dados (treino + teste) antes do split -> leakage
X_all_sc = scaler.fit_transform(X)  # fit no dataset inteiro, incluindo o "teste"
sel_leak = SelectKBest(f_classif, k=10)
X_all_sel = sel_leak.fit_transform(X_all_sc, y)  # o seletor já "viu" o teste
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(
    X_all_sel, y, test_size=0.3, random_state=42)
clf_leak = LogisticRegression(max_iter=10000).fit(X_tr_leak, y_tr_leak)
acc_leak = accuracy_score(y_te_leak, clf_leak.predict(X_te_leak))

print(f'Acurácia CORRETA (fit só no treino): {acc_ok:.4f}')
print(f'Acurácia COM LEAKAGE (fit em tudo):  {acc_leak:.4f}')
print('\nMesmo se os números ficarem próximos aqui, a versão com leakage nunca deve')
print('ser usada: o processo de seleção "espiou" o teste, então essa acurácia não é')
print('uma estimativa confiável do desempenho em dados novos.')

## 2.4 Comparando métodos filter (slide 38)

Repete o pipeline do slide 35, mas comparando `ANOVA (f_classif)` vs `Mutual Information`
lado a lado.

In [ ]:
results_filter = {}
for name, scorer in [('ANOVA', f_classif), ('MI', mutual_info_classif)]:
    sel_cmp = SelectKBest(scorer, k=10)
    Xtr = sel_cmp.fit_transform(X_train_sc, y_train)
    Xte = sel_cmp.transform(X_test_sc)
    clf_cmp = LogisticRegression(max_iter=10000)
    clf_cmp.fit(Xtr, y_train)
    acc = accuracy_score(y_test, clf_cmp.predict(Xte))
    results_filter[name] = acc
    print(f'{name:8s} -> Acurácia: {acc:.4f}')

## 2.5 Combinando filtros (slide 39)

Filtros podem ser encadeados dentro de um único `Pipeline`: primeiro `VarianceThreshold`
(remove features quase constantes), depois `SelectKBest` (seleciona as mais relevantes).
Usar `Pipeline` também evita data leakage automaticamente, pois `fit`/`transform` são
geridos corretamente em cada fold/split.

In [ ]:
pipe_combo = Pipeline([
    ('variance', VarianceThreshold(threshold=0.1)),
    ('kbest', SelectKBest(f_classif, k=10)),
    ('clf', LogisticRegression(max_iter=10000))
])

pipe_combo.fit(X_train_sc, y_train)
acc_combo = pipe_combo.score(X_test_sc, y_test)
print(f'Pipeline Filter combinado -> Acurácia: {acc_combo:.4f}')

---
# Bloco 3 — Métodos Wrapper

Diferente dos filtros, métodos **wrapper** usam um modelo real, treinado e reavaliado
repetidamente, para encontrar o melhor subconjunto de features. São mais precisos (captam
interações entre variáveis) mas computacionalmente mais caros.

## 3.1 Recursive Feature Elimination — RFE (slide 49)

Treina um modelo (`LogisticRegression`), remove a feature menos importante, e repete o
processo iterativamente (`step=1`) até restar o número desejado de features
(`n_features_to_select=10`). `rfe.ranking_` mostra a ordem de eliminação (1 = mais
importante).

In [ ]:
model_rfe = LogisticRegression(max_iter=10000)
rfe = RFE(model_rfe, n_features_to_select=10, step=1)
rfe.fit(X_train_sc, y_train)

selected_rfe = X.columns[rfe.support_]
print('Features selecionadas (RFE):', selected_rfe.tolist())
print('Ranking:', rfe.ranking_)

acc_rfe = rfe.score(X_test_sc, y_test)
print(f'Acurácia com RFE: {acc_rfe:.4f}')

## 3.2 RFECV — RFE com validação cruzada (slide 50)

Evolução do RFE: em vez de você fixar manualmente `n_features_to_select`, o `RFECV` testa
diferentes tamanhos de subconjunto usando validação cruzada (`cv=5`) e escolhe o que
maximiza a métrica de interesse. Mais robusto, porém mais custoso (é o método mais lento
desta aula).

In [ ]:
rfecv = RFECV(
    estimator=LogisticRegression(max_iter=10000),
    step=1,
    cv=5,
    scoring='accuracy',
    min_features_to_select=1
)
rfecv.fit(X_train_sc, y_train)

print(f'Número ótimo de features: {rfecv.n_features_}')
print(f'Features selecionadas: {X.columns[rfecv.support_].tolist()}')
print(f'Acurácia: {rfecv.score(X_test_sc, y_test):.4f}')

---
# Bloco 4 — Métodos Embedded

A seleção de features acontece "de graça" durante o próprio treinamento do modelo —
mais eficiente que wrapper, mas específica ao tipo de modelo usado.

## 4.1 Regularização L1 — Lasso (slide 54)

A penalização L1 (`penalty='l1'`) na `LogisticRegression` zera automaticamente os
coeficientes de features pouco relevantes (efeito de esparsidade do Lasso, controlado pelo
hiperparâmetro `C`). `SelectFromModel(..., prefit=True)` extrai apenas as features com
coeficiente diferente de zero.

In [ ]:
lr_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, max_iter=10000)
lr_l1.fit(X_train_sc, y_train)

sel_l1 = SelectFromModel(lr_l1, prefit=True)
X_tr_l1 = sel_l1.transform(X_train_sc)
X_te_l1 = sel_l1.transform(X_test_sc)

print(f'Features mantidas: {X_tr_l1.shape[1]} de {X_train_sc.shape[1]}')
print(f'Features zeradas: {(lr_l1.coef_[0] == 0).sum()}')

## 4.2 Feature Importance — Random Forest (slide 56)

O atributo nativo `rf.feature_importances_` mede a redução média de impureza (Gini)
proporcionada por cada feature nas árvores da floresta. `SelectFromModel(..., threshold='mean')`
seleciona automaticamente as features acima da importância média.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sc, y_train)

importances_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Top 10 features (Random Forest):')
print(importances_rf.head(10))

sel_rf = SelectFromModel(rf, prefit=True, threshold='mean')
X_tr_emb = sel_rf.transform(X_train_sc)
X_te_emb = sel_rf.transform(X_test_sc)
print(f'\nFeatures acima da média: {X_tr_emb.shape[1]}')

## 4.3 Feature Importance — XGBoost (slide 57)

Mesma ideia do Random Forest, agora com `XGBClassifier`. O XGBoost também oferece outros
critérios de importância (`gain`, `weight`, `cover`), mas aqui usamos o padrão
`feature_importances_` para manter a mesma interface (`SelectFromModel`) usada nos outros
métodos embedded.

In [ ]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_clf.fit(X_train_sc, y_train)

importances_xgb = pd.Series(xgb_clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Top 10 features (XGBoost):')
print(importances_xgb.head(10))

sel_xgb = SelectFromModel(xgb_clf, prefit=True)
X_tr_xgb = sel_xgb.transform(X_train_sc)
X_te_xgb = sel_xgb.transform(X_test_sc)
print(f'\nFeatures selecionadas: {X_tr_xgb.shape[1]}')

## 4.4 Permutation Importance (slide 58)

Técnica **model-agnostic** (funciona com qualquer modelo já treinado): mede a queda de
performance ao embaralhar aleatoriamente os valores de cada feature no conjunto de
**teste**, repetindo `n_repeats=10` vezes para estabilidade. Diferente da importância nativa
de árvores (enviesada por features de alta cardinalidade), é calculada diretamente sobre a
métrica de interesse e costuma ser mais confiável.

In [ ]:
perm_result = permutation_importance(
    rf, X_test_sc, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

perm_importances = pd.Series(perm_result.importances_mean, index=X.columns).sort_values(ascending=False)
print('Top 10 (Permutation Importance):')
print(perm_importances.head(10))

---
# Bloco 5 — Comparação final entre as três famílias

Fechamos comparando, lado a lado, um representante de cada família — **Filter (ANOVA)**,
**Wrapper (RFE)**, **Embedded (Random Forest)** e **Embedded (L1)** — tanto em **acurácia**
quanto em **tempo de execução**, usando sempre o mesmo classificador final
(`LogisticRegression`) para isolar o efeito da seleção de features.

> No material original (slide 60) esta célula estava incompleta — faltavam as variáveis
> `X_tr_filt`/`X_te_filt` (Filter) e `X_tr_wrap`/`X_te_wrap` (Wrapper), e a lista `methods`
> usada na medição de tempo nunca era definida. Abaixo está a versão corrigida e executável,
> reaproveitando o que já foi calculado nas células anteriores (`X_tr_l1`/`X_te_l1` do
> Lasso e `X_tr_emb`/`X_te_emb` do Random Forest).

In [ ]:
# Conjuntos "Filter" (ANOVA, slide 35) e "Wrapper" (RFE, slide 49)
X_tr_filt = sel.transform(X_train_sc)
X_te_filt = sel.transform(X_test_sc)

X_tr_wrap = rfe.transform(X_train_sc)
X_te_wrap = rfe.transform(X_test_sc)

results = {}
for name, Xtr, Xte in [
    ('Filter (ANOVA)', X_tr_filt, X_te_filt),
    ('Wrapper (RFE)',  X_tr_wrap, X_te_wrap),
    ('Embedded (RF)',  X_tr_emb,  X_te_emb),
    ('Embedded (L1)',  X_tr_l1,   X_te_l1),
]:
    clf_final = LogisticRegression(max_iter=10000)
    clf_final.fit(Xtr, y_train)
    acc = accuracy_score(y_test, clf_final.predict(Xte))
    results[name] = acc
    print(f'{name:20s} -> Acurácia: {acc:.4f}')

In [ ]:
# Tempo de execução de cada método (fit do seletor, não do classificador final)
methods = [
    ('Filter (ANOVA)', SelectKBest(f_classif, k=10)),
    ('Wrapper (RFE)',  RFE(LogisticRegression(max_iter=10000), n_features_to_select=10)),
    ('Embedded (RF)',  SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), threshold='mean')),
]

for name, sel_func in methods:
    start = time.time()
    sel_func.fit(X_train_sc, y_train)
    elapsed = time.time() - start
    print(f'{name:20s} -> Tempo: {elapsed:.3f}s')

**O que observar no resultado acima:**

- **Filter** costuma ser o mais rápido — não treina nenhum modelo, só calcula estatísticas.
- **Wrapper (RFE)** é o mais lento — retreina o modelo a cada feature removida.
- **Embedded** fica no meio-termo — aproveita um único treinamento do modelo final (ou de um modelo auxiliar) para obter a seleção.
- As acurácias tendem a ficar próximas neste dataset (é limpo e bem separável), mas o
  **custo computacional** e a **capacidade de capturar interações entre features** variam
  bastante entre os métodos — essa é a mensagem central da aula: não existe um método
  universalmente melhor, a escolha depende do trade-off entre custo, interpretabilidade e
  necessidade de capturar interações.

---
# Resumo — progressão lógica da aula

1. **Estatística univariada simples** (Pearson → Spearman → Chi² → Mutual Information):
   parte de correlação linear básica e generaliza para relações monotônicas e depois
   não-lineares, sempre uma feature por vez, sem nenhum modelo de ML.
2. **Filtros não-supervisionados e consolidação** (Variance Threshold → SelectKBest):
   acrescenta um filtro que ignora o alvo e unifica os métodos estatísticos numa única
   interface do sklearn.
3. **Aplicação prática e boas práticas** (pipeline completo → visualização → data leakage →
   comparação → combinação): sai da teoria isolada para um fluxo real de treino/teste,
   com ênfase no erro mais comum (vazamento de dados).
4. **Wrapper** (RFE → RFECV): sofisticação maior — considera o modelo final e interações
   entre variáveis, ao custo de retreinar repetidamente.
5. **Embedded** (L1/Lasso → Random Forest → XGBoost → Permutation Importance): seleção
   "de graça" durante o treino, ou medida de forma agnóstica ao modelo (permutação).
6. **Síntese comparativa**: acurácia vs. tempo entre representantes das três famílias.

Essa progressão prepara o terreno para os próximos temas da disciplina: otimização de
hiperparâmetros (aula 2), interpretabilidade com SHAP (aula 3), métricas de avaliação
(aula 4) e viés/variância (aula 5).

---
# Exercícios

Agora é sua vez. Os exercícios abaixo reforçam cada bloco da aula usando o mesmo dataset
(`breast_cancer`) e as variáveis já criadas nas células anteriores. Tente resolver cada um
na célula "**Sua vez**" antes de olhar a célula "**Solução**" logo abaixo.

Todos os exercícios reaproveitam `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test`,
`X_train_sc`, `X_test_sc` e as variáveis calculadas ao longo do notebook (ex: `acc_filter_pipeline`,
`selected_rfe`, `perm_importances`) — rode o notebook em ordem, do início, antes de chegar aqui.

## Exercício 1 — Filter: menos é mais?

No slide 35 selecionamos as **10** melhores features por ANOVA e treinamos uma
`LogisticRegression`, obtendo uma acurácia guardada em `acc_filter_pipeline`.

**Tarefa:** repita o mesmo pipeline (`SelectKBest(f_classif, ...)` → `LogisticRegression`)
selecionando apenas as **5** melhores features. A acurácia cai muito em relação ao `k=10`?
O que isso sugere sobre quantas features deste dataset realmente carregam sinal relevante?

In [ ]:
# Sua vez: crie sel5 = SelectKBest(f_classif, k=5), ajuste no treino, transforme o teste,
# treine uma LogisticRegression e calcule a acurácia. Compare com acc_filter_pipeline.


**Solução**

In [ ]:
sel5 = SelectKBest(f_classif, k=5)
Xtr5 = sel5.fit_transform(X_train_sc, y_train)
Xte5 = sel5.transform(X_test_sc)

clf5 = LogisticRegression(max_iter=10000)
clf5.fit(Xtr5, y_train)
acc5 = accuracy_score(y_test, clf5.predict(Xte5))

print(f'Acurácia com k=5:  {acc5:.4f}')
print(f'Acurácia com k=10: {acc_filter_pipeline:.4f}')
print('\nSe a queda for pequena, é sinal de que boa parte da informação está concentrada')
print('em poucas features fortemente correlacionadas com o alvo (esperado neste dataset,')
print('já que várias features de "mean", "worst" e "error" medem a mesma grandeza).')

## Exercício 2 — Encontre o data leakage

O código abaixo (baseado no slide 37) tem um bug de vazamento de dados. Rode a célula,
depois **encontre e corrija o erro** na célula "Sua vez".

In [ ]:
# Código COM BUG — rode e depois corrija na célula de baixo
scaler_bug = StandardScaler()
X_bug_scaled = scaler_bug.fit_transform(X)          # <- fit no dataset inteiro

sel_bug = SelectKBest(f_classif, k=10)
X_bug_sel = sel_bug.fit_transform(X_bug_scaled, y)   # <- fit no dataset inteiro (com y inteiro)

Xtr_bug, Xte_bug, ytr_bug, yte_bug = train_test_split(
    X_bug_sel, y, test_size=0.3, random_state=42)     # split só depois de já ter "visto" tudo

clf_bug = LogisticRegression(max_iter=10000)
clf_bug.fit(Xtr_bug, ytr_bug)
print(f'Acurácia (com leakage): {accuracy_score(yte_bug, clf_bug.predict(Xte_bug)):.4f}')

**Sua vez:** reescreva o pipeline acima na ordem correta — `train_test_split` **primeiro**,
depois `fit_transform` do scaler e do seletor **só no treino**, e `transform` (sem fit) no
teste.

In [ ]:
# Sua vez: corrija a ordem das operações para eliminar o leakage.


**Solução**

In [ ]:
# CORRETO: split primeiro, fit só no treino
Xtr_fix, Xte_fix, ytr_fix, yte_fix = train_test_split(X, y, test_size=0.3, random_state=42)

scaler_fix = StandardScaler()
Xtr_fix_sc = scaler_fix.fit_transform(Xtr_fix)
Xte_fix_sc = scaler_fix.transform(Xte_fix)

sel_fix = SelectKBest(f_classif, k=10)
Xtr_fix_sel = sel_fix.fit_transform(Xtr_fix_sc, ytr_fix)
Xte_fix_sel = sel_fix.transform(Xte_fix_sc)

clf_fix = LogisticRegression(max_iter=10000)
clf_fix.fit(Xtr_fix_sel, ytr_fix)
print(f'Acurácia (sem leakage): {accuracy_score(yte_fix, clf_fix.predict(Xte_fix_sel)):.4f}')
print('\nO erro estava em chamar fit_transform do scaler e do SelectKBest em X/y')
print('completos, ANTES do train_test_split — o teste "vazava" para dentro do ajuste.')

## Exercício 3 — Wrapper com outro estimador

No slide 49 usamos `RFE` com `LogisticRegression`. O RFE funciona com qualquer estimador
que exponha `coef_` ou `feature_importances_`.

**Tarefa:** rode o `RFE` usando `DecisionTreeClassifier` como estimador (mesmo
`n_features_to_select=10`) e compare o conjunto de features escolhidas com o obtido pela
`LogisticRegression` (variável `selected_rfe`, calculada no slide 49). Quantas features
coincidem?

In [ ]:
# Sua vez: from sklearn.tree import DecisionTreeClassifier
# crie um RFE com DecisionTreeClassifier(random_state=42), ajuste, e compare
# set(X.columns[rfe_tree.support_]) com set(selected_rfe).


**Solução**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

rfe_tree = RFE(DecisionTreeClassifier(random_state=42), n_features_to_select=10, step=1)
rfe_tree.fit(X_train_sc, y_train)

selecionadas_tree = set(X.columns[rfe_tree.support_])
selecionadas_logreg = set(selected_rfe)
intersecao = selecionadas_tree & selecionadas_logreg

print('Selecionadas com Decision Tree:', sorted(selecionadas_tree))
print('Selecionadas com Logistic Regression:', sorted(selecionadas_logreg))
print(f'\nFeatures em comum: {len(intersecao)} de 10')
print(sorted(intersecao))
print('\nA interseção raramente é de 100%: cada modelo "enxerga" relevância de um jeito')
print('diferente — árvores capturam interações e cortes não-lineares, a regressão')
print('logística captura relações lineares no espaço padronizado.')

## Exercício 4 — O efeito do hiperparâmetro C no Lasso

No slide 54 usamos `C=0.1` na `LogisticRegression(penalty='l1')`. Lembrando: `C` é o
inverso da força de regularização — **quanto menor `C`, mais forte a penalização**, e mais
coeficientes são zerados.

**Tarefa:** teste `C` em `[0.01, 0.1, 1, 10]` e imprima, para cada valor, quantas features
sobrevivem (coeficiente ≠ 0). O que acontece nos extremos?

In [ ]:
# Sua vez: para cada C em [0.01, 0.1, 1, 10], treine LogisticRegression(penalty='l1',
# solver='liblinear', C=C, max_iter=10000) em X_train_sc/y_train e conte quantos
# coeficientes são diferentes de zero.


**Solução**

In [ ]:
for C in [0.01, 0.1, 1, 10]:
    lr_c = LogisticRegression(penalty='l1', solver='liblinear', C=C, max_iter=10000)
    lr_c.fit(X_train_sc, y_train)
    n_zero = int((lr_c.coef_[0] == 0).sum())
    n_keep = X_train_sc.shape[1] - n_zero
    print(f'C={C:<6} -> features mantidas: {n_keep:2d} | zeradas: {n_zero:2d}')

print('\nC muito pequeno (ex: 0.01): penalização forte -> quase tudo zerado, modelo simples')
print('demais (risco de underfitting). C muito grande (ex: 10): penalização fraca -> quase')
print('nenhuma feature é removida, o Lasso deixa de funcionar como seletor de features.')

## Exercício 5 — Permutation Importance com outro modelo

No slide 58 calculamos a permutation importance usando o `RandomForestClassifier` (`rf`),
guardada em `perm_importances`.

**Tarefa:** calcule a permutation importance usando a `LogisticRegression` treinada com
todas as features (`X_train_sc`/`y_train`) e compare o **top 5** com o do Random Forest.
O ranking muda muito entre os dois modelos?

In [ ]:
# Sua vez: treine uma LogisticRegression em X_train_sc/y_train, rode permutation_importance
# nela sobre X_test_sc/y_test (n_repeats=10, random_state=42), e compare o top 5 com
# perm_importances.head(5).


**Solução**

In [ ]:
clf_full = LogisticRegression(max_iter=10000)
clf_full.fit(X_train_sc, y_train)

perm_lr = permutation_importance(
    clf_full, X_test_sc, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
perm_lr_series = pd.Series(perm_lr.importances_mean, index=X.columns).sort_values(ascending=False)

print('Top 5 — Permutation Importance (Random Forest):')
print(perm_importances.head(5))
print('\nTop 5 — Permutation Importance (Logistic Regression):')
print(perm_lr_series.head(5))

---
# Aplicação a outro dataset — Wine

Para confirmar que o que você aprendeu não é específico do `breast_cancer`, vamos repetir
os pontos-chave da aula (baseline → filter → wrapper → embedded → comparação) no dataset
**Wine**, também nativo do sklearn (não precisa de download).

**Por que o Wine é um bom segundo exemplo:**

- **Multi-classe** (3 cultivares de uva), diferente do `breast_cancer` que é binário — testa
  se os métodos generalizam além de classificação binária.
- **Bem menos dados**: 178 amostras e 13 features (contra 569 amostras e 30 features do
  `breast_cancer`) — dataset pequeno, mais sensível a overfitting, então a seleção de
  features tende a ajudar mais (ou pelo menos a mudar de forma mais visível).

## Setup — Wine

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
Xw = pd.DataFrame(wine.data, columns=wine.feature_names)
yw = pd.Series(wine.target)

print(f'Shape: {Xw.shape} | Classes: {sorted(yw.unique())} | Nomes: {list(wine.target_names)}')

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.3, random_state=42, stratify=yw)

scaler_w = StandardScaler()
Xw_train_sc = scaler_w.fit_transform(Xw_train)
Xw_test_sc = scaler_w.transform(Xw_test)

print(f'Treino: {Xw_train.shape} | Teste: {Xw_test.shape}')

## Baseline — todas as features

Antes de selecionar qualquer feature, medimos a acurácia usando as 13 features originais.
É a referência contra a qual comparamos os métodos de seleção.

In [ ]:
clf_baseline_w = LogisticRegression(max_iter=10000)
clf_baseline_w.fit(Xw_train_sc, yw_train)
acc_baseline_w = accuracy_score(yw_test, clf_baseline_w.predict(Xw_test_sc))
print(f'Baseline (13 features) -> Acurácia: {acc_baseline_w:.4f}')

## Filter — ANOVA (SelectKBest)

Com apenas 13 features, faz sentido selecionar um número menor, por exemplo `k=5`.

In [ ]:
sel_w = SelectKBest(f_classif, k=5)
Xw_tr_filt = sel_w.fit_transform(Xw_train_sc, yw_train)
Xw_te_filt = sel_w.transform(Xw_test_sc)

top5_w = pd.Series(sel_w.scores_, index=Xw.columns).sort_values(ascending=False).head(5)
print('Top 5 features (ANOVA):', top5_w.index.tolist())

clf_filt_w = LogisticRegression(max_iter=10000)
clf_filt_w.fit(Xw_tr_filt, yw_train)
acc_filt_w = accuracy_score(yw_test, clf_filt_w.predict(Xw_te_filt))
print(f'Filter (ANOVA, k=5) -> Acurácia: {acc_filt_w:.4f}')

## Wrapper — RFE

In [ ]:
rfe_w = RFE(LogisticRegression(max_iter=10000), n_features_to_select=5, step=1)
rfe_w.fit(Xw_train_sc, yw_train)

Xw_tr_wrap = rfe_w.transform(Xw_train_sc)
Xw_te_wrap = rfe_w.transform(Xw_test_sc)

print('Features selecionadas (RFE):', Xw.columns[rfe_w.support_].tolist())
acc_wrap_w = accuracy_score(yw_test, rfe_w.predict(Xw_test_sc))
print(f'Wrapper (RFE, 5 features) -> Acurácia: {acc_wrap_w:.4f}')

## Embedded — Random Forest e L1

In [ ]:
# Embedded: Random Forest
rf_w = RandomForestClassifier(n_estimators=100, random_state=42)
rf_w.fit(Xw_train_sc, yw_train)

importances_w = pd.Series(rf_w.feature_importances_, index=Xw.columns).sort_values(ascending=False)
print('Top 5 features (Random Forest):')
print(importances_w.head(5))

sel_rf_w = SelectFromModel(rf_w, prefit=True, threshold='mean')
Xw_tr_emb = sel_rf_w.transform(Xw_train_sc)
Xw_te_emb = sel_rf_w.transform(Xw_test_sc)

clf_emb_w = LogisticRegression(max_iter=10000)
clf_emb_w.fit(Xw_tr_emb, yw_train)
acc_emb_w = accuracy_score(yw_test, clf_emb_w.predict(Xw_te_emb))
print(f'\nEmbedded (RF, {Xw_tr_emb.shape[1]} features) -> Acurácia: {acc_emb_w:.4f}')

In [ ]:
# Embedded: L1 / Lasso
# 'liblinear' só suporta classificação binária; Wine tem 3 classes, então usamos 'saga'
# (também suporta penalty='l1' e lida nativamente com multiclasse)
lr_l1_w = LogisticRegression(penalty='l1', solver='saga', C=0.5, max_iter=10000)
lr_l1_w.fit(Xw_train_sc, yw_train)

sel_l1_w = SelectFromModel(lr_l1_w, prefit=True)
Xw_tr_l1 = sel_l1_w.transform(Xw_train_sc)
Xw_te_l1 = sel_l1_w.transform(Xw_test_sc)

clf_l1_w = LogisticRegression(max_iter=10000)
clf_l1_w.fit(Xw_tr_l1, yw_train)
acc_l1_w = accuracy_score(yw_test, clf_l1_w.predict(Xw_te_l1))
print(f'Embedded (L1, {Xw_tr_l1.shape[1]} features) -> Acurácia: {acc_l1_w:.4f}')

## Comparação final — Wine

In [ ]:
results_wine = {
    'Baseline (13 features)': acc_baseline_w,
    'Filter (ANOVA, 5)': acc_filt_w,
    'Wrapper (RFE, 5)': acc_wrap_w,
    'Embedded (RF)': acc_emb_w,
    'Embedded (L1)': acc_l1_w,
}

print(f'{"Método":25s} Acurácia')
print('-' * 36)
for name, acc in results_wine.items():
    print(f'{name:25s} {acc:.4f}')

**O que comparar com o `breast_cancer`:**

- O dataset Wine tem só 178 amostras — bem menos que as 569 do `breast_cancer` — então é
  mais fácil um método de seleção "acertar" quase todas as classes corretamente mesmo
  removendo bastante features (o problema é mais simples/separável, com 3 classes bem
  distintas quimicamente).
- Repare que os mesmos métodos (`SelectKBest`, `RFE`, `SelectFromModel`) funcionaram sem
  nenhuma alteração de código, só trocando o dataset de entrada — essa é a vantagem de
  usar as APIs padronizadas do scikit-learn: o pipeline de seleção de features é
  praticamente **plug-and-play** entre datasets diferentes, contanto que o problema seja
  de classificação supervisionada.